In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Using project root:", project_root)
print("Ensured project root on sys.path")

Using project root: /Users/VanshitaS/Desktop/UVA - Period 4/Big Data/Big-Data/big_data_assignment
Ensured project root on sys.path


### Environment setup

Before running this notebook, install the project dependencies in a terminal from the project root (`big_data_assignment`):

```bash
cd ~/Desktop/Big-Data/big_data_assignment
pip install -r requirements.txt
```

After installing, restart the Jupyter kernel and re-run the first environment cell.

# IMDB Train Data - Completeness & Quality Checks

This notebook loads the IMDB training data and performs:
- **Datatype check per column**
- **Missing value analysis**
- **Disguised missing data detection**
- **Checks for negative numeric values in `runtimeMinutes` and `numVotes`**


In [6]:
BASE_PATH = Path("/Users/VanshitaS/Desktop/UVA - Period 4/Big Data/Big-Data/big_data_assignment/members/Vanshita/imdb")  # adjust if needed

csv_files      = sorted(BASE_PATH.glob("train-*.csv"))
writing_json   = BASE_PATH / "writing.json"
directing_json = BASE_PATH / "directing.json"

In [7]:
import pandas as pd
from pathlib import Path

# Path to the IMDB project directory
# base_path = Path('..') / 'big-data-course-2024-projects' / 'imdb'

# Collect and load all train-*.csv files
train_files = sorted([f for f in BASE_PATH.glob('train-*.csv') if f.is_file()])
dfs = []
for f in train_files:
    print(f'Loading {f.name}...')
    dfs.append(pd.read_csv(f))

df_train = pd.concat(dfs, ignore_index=True)
print(f'Total rows in combined train data: {len(df_train)}')
df_train.head()

Loading train-1.csv...
Loading train-2.csv...
Loading train-3.csv...
Loading train-4.csv...
Loading train-5.csv...
Loading train-6.csv...
Loading train-7.csv...
Loading train-8.csv...
Total rows in combined train data: 7959


,Unnamed: 0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
0,4,tt0010600,The Doll,Die Puppe,1919,\N,66,1898.0,True
1,7,tt0011841,Way Down East,Way Down East,1920,\N,145,5376.0,True
2,9,tt0012494,Déstiny,Der müde Tod,1921,\N,97,5842.0,True
3,25,tt0015163,The Navigator,The Navigator,1924,\N,59,9652.0,True
4,38,tt0016220,The Phantom of the Opera,The Phantom of the Opera,1925,\N,93,17887.0,True


In [8]:
# 1. Datatype check per column
print("Column dtypes in df_train:\n")
print(df_train.dtypes)


Column dtypes in df_train:

Unnamed: 0          int64
tconst             object
primaryTitle       object
originalTitle      object
startYear          object
endYear            object
runtimeMinutes     object
numVotes          float64
label                bool
dtype: object


**Interpretation of MCAR-style diagnostics**

- **Missingness indicator correlations**
  - **startYear vs endYear (−1.0)**: When `startYear` is missing, `endYear` is almost always observed, and vice versa. That reflects the data structure: movies usually have only `startYear`; series/ongoing titles often have `endYear` and sometimes missing `startYear`. So this missingness is **not MCAR** — it is **structural** (MAR: missingness depends on “type” of title).
  - The other indicator pairs have correlations near 0 (e.g. −0.013, 0.007, −0.003), so those missingness patterns are **not strongly related to each other**.

- **Group-wise means (missing vs observed)**
  - **startYear / endYear**: When one is missing, the other is observed and has a different profile (e.g. mean `numVotes` ~29k vs ~34k). Again, this points to **not MCAR** (missingness tied to whether the title is a movie vs series).
  - **runtimeMinutes**: When runtime is missing, mean `numVotes` is much lower (~1.7k vs ~29.6k) and mean `startYear` is later (~2013 vs ~1998). So **runtime missingness is related to observed variables** (e.g. less popular or newer titles) → **not MCAR**, likely MAR.
  - **numVotes**: Means of `startYear`, `endYear`, and `runtimeMinutes` are very similar between missing and observed groups. **numVotes missingness is the most consistent with MCAR** among the four columns.

**Conclusion:** After treating `\N` as NaN, the missingness in **startYear**, **endYear**, and **runtimeMinutes** appears **not MCAR** (structural or MAR). **numVotes** missingness is the closest to MCAR. For downstream analysis, consider MAR-aware methods (e.g. include predictors of missingness) rather than assuming MCAR everywhere.

In [9]:
import numpy as np
from collections import Counter

def character_entropy(s: str) -> float:
    """
    Shannon entropy of the character distribution in a string.
    Short/simple strings like 'NA', '\\N', 'N/A' score near 0.
    Richer strings like 'tt0032138' or 'The Wizard of Oz' score higher.
    Returns 0.0 for empty or single-character strings.
    """
    if not s or len(s) <= 1:
        return 0.0
    counts = Counter(s)
    total = len(s)
    probs = [c / total for c in counts.values()]
    return -sum(p * np.log2(p) for p in probs)


def detect_disguised_nulls(df, entropy_threshold=1.0, freq_threshold=0.05):
    """
    For each object column, flag values that look like sentinel/disguised nulls
    based on low character entropy AND low frequency.

    Parameters
    ----------
    entropy_threshold : float
        Max character entropy to consider a value "suspiciously simple".
        - "\\N"  → entropy ≈ 0.0
        - "NA"   → entropy ≈ 0.0  
        - "N/A"  → entropy ≈ 1.58
        - "null" → entropy ≈ 1.5
        - "tt0032138" → entropy ≈ 2.75
        A threshold of ~1.5 catches most sentinel patterns.

    freq_threshold : float
        Max fraction of rows a suspicious value can represent.
        Avoids flagging a dominant real value that happens to be simple.
    """
    token_hits = {}

    for col in df.columns:
        if df[col].dtype != "O":
            continue

        vc = df[col].value_counts(dropna=False, normalize=True)  # gives proportions
        n_rows = len(df)

        for val, proportion in vc.items():
            # Treat NaN separately — always suspicious
            if val is np.nan or val is None:
                continue  # real nulls caught by missing-value check

            val_str = str(val).strip()

            entropy = character_entropy(val_str)
            is_low_entropy = entropy <= entropy_threshold
            is_low_frequency = proportion <= freq_threshold

            if is_low_entropy and is_low_frequency:
                count = int(round(proportion * n_rows))
                token_hits[f"{col}:{repr(val_str)}"] = {
                    "count": count,
                    "pct": round(proportion * 100, 2),
                    "entropy": round(entropy, 3)
                }

    return token_hits

In [10]:
def run_quality_audit(df, label="dataset"):
    """
    Single entry point that calls all of Lisa's checks.
    Returns a results dict with pass/warn per check.
    """
    import numpy as np
    results = {}

    # disguised null imputation 
    df_clean = df.copy()
    for col in df_clean.columns:
        if df_clean[col].dtype == "O":
            df_clean[col] = df_clean[col].replace("\\N", np.nan)

    # missing values 
    raw_nulls = df.isna().sum()
    clean_nulls = df_clean.isna().sum()
    disguised = (clean_nulls - raw_nulls)
    results["missing"] = disguised[disguised > 0].to_dict()

    # Disguised tokens — detected via entropy, not hardcoded list
    results["disguised_tokens"] = detect_disguised_nulls(
        df, 
        entropy_threshold=1.5,   # tune this — higher catches more
        freq_threshold=0.05      # ignore values appearing in >5% of rows
    )

    # suspicious numeric rules
    SUSPICIOUS_RULES = {
        "runtimeMinutes": lambda s: pd.to_numeric(s, errors="coerce") <= 0,
        "startYear":      lambda s: pd.to_numeric(s, errors="coerce") < 1888,
        "numVotes":       lambda s: pd.to_numeric(s, errors="coerce") <= 0,
    }
    suspicious = {}
    for col, rule in SUSPICIOUS_RULES.items():
        if col in df.columns:
            count = int(rule(df[col]).fillna(False).sum())
            if count > 0:
                suspicious[col] = count
    results["suspicious_values"] = suspicious

    # validity (endYear < startYear)
    sy = pd.to_numeric(df_clean.get("startYear"), errors="coerce")
    ey = pd.to_numeric(df_clean.get("endYear"),   errors="coerce")
    if sy is not None and ey is not None:
        both = sy.notna() & ey.notna()
        results["endYear_before_startYear"] = int((both & (ey < sy)).sum())

    # duplicates 
    results["exact_duplicate_rows"] = int(df.duplicated().sum())
    results["duplicate_tconst"] = int(
        df["tconst"].duplicated().sum() if "tconst" in df.columns else 0
    )

    # Print summary 
    print(f"\n Quality Audit: {label} ({len(df):,} rows)\n")
    for check, val in results.items():
        if isinstance(val, dict):
            status = "✅ PASS" if not val else "⚠  WARN"
            print(f"{status}  {check}: {val if val else 'none'}")
        else:
            status = "✅ PASS" if val == 0 else "⚠  WARN"
            print(f"{status}  {check}: {val}")

    return results

# Run it:
audit_results = run_quality_audit(df_train, label="imdb_train (raw)")

# Run again after cleaning to verify fixes:
# audit_results_clean = run_quality_audit(df_train_imputed, label="imdb_train (cleaned)")


 Quality Audit: imdb_train (raw) (7,959 rows)

⚠  WARN  missing: {'startYear': 786, 'endYear': 7173, 'runtimeMinutes': 13}
⚠  WARN  disguised_tokens: {"tconst:'tt0090009'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.436)}, "tconst:'tt1119191'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.436)}, "primaryTitle:'Solo'": {'count': 3, 'pct': 0.04, 'entropy': np.float64(1.5)}, "primaryTitle:'Coco'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Guru'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Z'": {'count': 2, 'pct': 0.03, 'entropy': 0.0}, "primaryTitle:'Deep'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Raat'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'Zulu'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'Lolo'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'R'": {'count': 1, 'pct': 0.01, 'entropy': 0.0}, "primaryTitle:'Ed'

In [11]:
# Step 1: Pull the table out of DuckDB into a DataFrame
import duckdb
from pathlib import Path

# Connect to the DuckDB database file in read-only mode to avoid lock conflicts
db_path = Path("/Users/VanshitaS/Desktop/UVA - Period 4/Big Data/Big-Data/big_data_assignment/members/Vanshita/imdb.duckdb")
con = duckdb.connect(str(db_path))

df_train = con.execute("SELECT * FROM imdb_train").fetchdf()
run_quality_audit(df_train, label="raw")


 Quality Audit: raw (7,959 rows)

⚠  WARN  missing: {'startYear': 786, 'endYear': 7173, 'runtimeMinutes': 13}
⚠  WARN  disguised_tokens: {"tconst:'tt0090009'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.436)}, "tconst:'tt1119191'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.436)}, "primaryTitle:'Solo'": {'count': 3, 'pct': 0.04, 'entropy': np.float64(1.5)}, "primaryTitle:'Coco'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Guru'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Z'": {'count': 2, 'pct': 0.03, 'entropy': 0.0}, "primaryTitle:'Deep'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)}, "primaryTitle:'Raat'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'Zulu'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'Lolo'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)}, "primaryTitle:'R'": {'count': 1, 'pct': 0.01, 'entropy': 0.0}, "primaryTitle:'Ed'": {'count': 

{'missing': {'startYear': 786, 'endYear': 7173, 'runtimeMinutes': 13},
 'disguised_tokens': {"tconst:'tt0090009'": {'count': 1,
   'pct': 0.01,
   'entropy': np.float64(1.436)},
  "tconst:'tt1119191'": {'count': 1,
   'pct': 0.01,
   'entropy': np.float64(1.436)},
  "primaryTitle:'Solo'": {'count': 3, 'pct': 0.04, 'entropy': np.float64(1.5)},
  "primaryTitle:'Coco'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)},
  "primaryTitle:'Guru'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)},
  "primaryTitle:'Z'": {'count': 2, 'pct': 0.03, 'entropy': 0.0},
  "primaryTitle:'Deep'": {'count': 2, 'pct': 0.03, 'entropy': np.float64(1.5)},
  "primaryTitle:'Raat'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)},
  "primaryTitle:'Zulu'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)},
  "primaryTitle:'Lolo'": {'count': 1, 'pct': 0.01, 'entropy': np.float64(1.5)},
  "primaryTitle:'R'": {'count': 1, 'pct': 0.01, 'entropy': 0.0},
  "primaryTitle:'Ed'": {'count': 1, 'pct': 0.

In [30]:
# e.g. after replacing \N with NULL in DuckDB
df_clean = con.execute("""
    SELECT * FROM imdb_train
    WHERE runtimeMinutes != '\\N'
""").fetchdf()
run_quality_audit(df_clean, label="after cleaning")


 Quality Audit: after cleaning (7,946 rows)

⚠  WARN  missing: {'startYear': 786, 'endYear': 7160}
⚠  WARN  disguised_tokens: {'startYear:\\N': 786, 'endYear:\\N': 7160}
✅ PASS  suspicious_values: none
✅ PASS  endYear_before_startYear: 0
✅ PASS  exact_duplicate_rows: 0
✅ PASS  duplicate_tconst: 0


{'missing': {'startYear': 786, 'endYear': 7160},
 'disguised_tokens': {'startYear:\\N': 786, 'endYear:\\N': 7160},
 'suspicious_values': {},
 'endYear_before_startYear': 0,
 'exact_duplicate_rows': 0,
 'duplicate_tconst': 0}